# Project Milestone 4
* Data Preparation: DSC 540-T302 
* Mallory Young
* Student #21356619

In [1]:
#importing libraries/packages I might need for this data transformation and/or cleansing
import requests
import locale 
locale.setlocale( locale.LC_ALL, '' )
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import json

### API Data Pull
The data pulled from the TMDB API is general film information and information about films with the highest revenue. The data pulled from the API is stored in a pandas dataframe named 'film_revenue_df'. 

In [2]:
#assigning my api key to a variable
#I have not 'hidden' my api key because I will not be publishing this code for others to see
api_key = '579428cf3535674a50921ff8907abdd4'

In [3]:
#using a .get() request to the TMDB API by concatenating 3 values:
#the TMBD url, my api key, and a sorting parameter to retrieve information about movies sorted by revenue in descending order
response = requests.get('https://api.themoviedb.org/3/discover/movie?api_key=' +  api_key + '&sort_by=revenue.desc')

#capturing the response from the API and converting it to a JSON format
highest_revenue_ever = response.json()

#accessing the 'results' key within the JSON data to extracts the list of movies with the highest revenue and store it in a variable
highest_revenue_films_ever = highest_revenue_ever['results']

In [4]:
#initializing a list of column names for the dataframe
columns = ['film', 'id', 'revenue', 'budget', 'release_date', 'original_language', 'genre_ids', 'popularity', 'vote_count', 'vote_average', 'overview', 'production_countries', 'status', 'tagline', 'runtime']

#creating an empty dataframe named 'film_revenue_df' with the specified columns
film_revenue_df = pd.DataFrame(columns=columns)

In [5]:
#a for loop that iterates through each movie in the highest_revenue_films_ever list 
for film in highest_revenue_films_ever:
    #making an API call to retrieve detailed information about the movie using its ID
    film_revenue = requests.get('https://api.themoviedb.org/3/movie/'+ str(film['id']) +'?api_key='+ api_key+'&language=en-US')
    #converting the API response to JSON format
    film_revenue = film_revenue.json()

    #if statement to check if a film's budget was greater than 100 million
    if film_revenue['budget'] > 100:
        #adding a new row to the dataframe with details of the movie if the budget condition is met
        film_revenue_df.loc[len(film_revenue_df)]=[film['title'], film['id'], film_revenue['revenue'], (film_revenue['budget'] * -1), film_revenue['release_date'], film['original_language'], film['genre_ids'], film['popularity'], film['vote_count'], film['vote_average'], film['overview'], film_revenue['production_countries'], film_revenue['status'], film_revenue['tagline'], film_revenue['runtime']]

In [6]:
#printing the number of rows and columns for the dataframe
film_revenue_df.shape

(20, 15)

In [7]:
#printing all 20 rows and 15 columns of the dataframe
pd.set_option('display.max_columns', None)
film_revenue_df

,film,id,revenue,budget,release_date,original_language,genre_ids,popularity,vote_count,vote_average,overview,production_countries,status,tagline,runtime
0,Avatar,19995,2923706026,-237000000,2009-12-15,en,"[28, 12, 14, 878]",111.323,31119,7.582,"In the 22nd century, a paraplegic Marine is di...","[{'iso_3166_1': 'US', 'name': 'United States o...",Released,Enter the world of Pandora.,162
1,Avengers: Endgame,299534,2799439100,-356000000,2019-04-24,en,"[12, 878, 28]",202.043,25178,8.253,After the devastating events of Avengers: Infi...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,Avenge the fallen.,181
2,Avatar: The Way of Water,76600,2320250281,-460000000,2022-12-14,en,"[878, 12, 28]",169.075,11554,7.622,Set more than a decade after the events of the...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,Return to Pandora.,192
3,Titanic,597,2264162353,-200000000,1997-11-18,en,"[18, 10749]",206.498,24922,7.906,101-year-old Rose DeWitt Bukater tells the sto...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,Nothing on Earth could come between them.,194
4,Star Wars: The Force Awakens,140607,2068223624,-245000000,2015-12-15,en,"[12, 28, 878]",90.923,19088,7.300,Thirty years after defeating the Galactic Empi...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,Every generation has a story.,136
5,Avengers: Infinity War,299536,2052415039,-300000000,2018-04-25,en,"[12, 28, 878]",370.572,29220,8.246,As the Avengers and their allies have continue...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,An entire universe. Once and for all.,149
6,Best Of Joy,1326885,2000000000,-1000,2024-12-25,en,"[18, 10402]",0.000,0,0.000,When his biggest show inches closer and closer...,[],Released,"It was the Best of Joy, it was the Worst of Joy",120
7,Spider-Man: No Way Home,634649,1921847111,-200000000,2021-12-15,en,"[28, 12, 878]",280.575,19693,8.000,Peter Parker is unmasked and no longer able to...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,The Multiverse unleashed.,148
8,Jurassic World,135397,1671537444,-150000000,2015-06-06,en,"[28, 12, 878, 53]",98.471,20079,6.689,Twenty-two years after the events of Jurassic ...,"[{'iso_3166_1': 'US', 'name': 'United States o...",Released,The park is open.,124
9,The Lion King,420818,1663000000,-260000000,2019-07-12,en,"[16, 12, 18, 10751]",118.265,9796,7.113,"Simba idolizes his father, King Mufasa, and ta...","[{'iso_3166_1': 'US', 'name': 'United States o...",Released,The King has Returned.,118


## Step 1: Finding Missing Values and Duplicates
Doing an API call from TMBD can return inconsistent and sometimes duplicate data. By checking for and handling duplicate values, we can catch potential inconsistencies in the API responses. We also want to make sure that the dataset is complete and address any possible missing values. 

In general, it is important to determine if a dataset contains missing or duplicate values for the following reasons: 
* **Data integrity:** It allows me to ensure the integrity of the chosen dataset and to understand the extent of missing/duplicate data so that I can take the appropriate actions to address it.
* **Data Analysis:** Missing/duplicate values can impact the accuracy and reliability of future data analysis.
* **Decision Making:** Incomplete data can lead to biased or inaccurate decision-making. By identifying missing values, I am able to make better decisions based on complete and reliable information.

In [8]:
#using the isnull() function to identify missing values
missing_values = film_revenue_df.isnull()
#using .sum() to count the total number of missing values in each column
missing_counts = missing_values.sum()
#displaying the total number of missing values in each column
print(missing_counts)
#Observation: There are no missing values in the 'film_revenue_df' dataframe

film                    0
id                      0
revenue                 0
budget                  0
release_date            0
original_language       0
genre_ids               0
popularity              0
vote_count              0
vote_average            0
overview                0
production_countries    0
status                  0
tagline                 0
runtime                 0
dtype: int64


**Observation:** While looking through the dataframe, I noticed that there is a blank space in the 'tagline' column for index #19. This did not get identified by .isnull(). This missing value needs to be handled, therefore I will fill the empty entry with N/A

In [9]:
#filling the empty space with 'N/A' 
film_revenue_df.loc[19, 'tagline'] = None

In [10]:
#creating an array of unique film titles that are duplicated in the DataFrame
#.duplicated() method with the subset='film' parameter checks for duplicate values in the "film" column
#the keep=False parameter marks all instances of the duplicated film titles as True
duplicate_titles = film_revenue_df[film_revenue_df.duplicated(subset='film', keep=False)]['film'].unique()
#printing the array of duplicated film titles
print(duplicate_titles)
#Observation: There are no duplicate values in the 'film_revenue_df' dataframe

[]


## Step 2: Replace & Rearrange Headers
Some of the headers are a bit vague and could be more precise. For example, the column headers 'runtime', 'revenue', and budget' do not specify units (such as minutes, or USD), and 'plot' would be a better description for the 'overview' column. I am also going to rearrange the headers so that similar information is closer together.

In [11]:
#displaying the column names to use when creating my new rearranged list of headers
film_revenue_df.columns

Index(['film', 'id', 'revenue', 'budget', 'release_date', 'original_language',
       'genre_ids', 'popularity', 'vote_count', 'vote_average', 'overview',
       'production_countries', 'status', 'tagline', 'runtime'],
      dtype='object')

In [12]:
#replacing the header (column) names using a list
new_col_names = ['film', 'id', 'revenue (USD)', 'budget (USD)', 'release_date', 'original_language',
       'genre_ids', 'popularity', 'vote_count', 'vote_average', 'plot',
       'production_countries', 'release_status', 'tagline', 'runtime (min)']
#assigning the new column names using the 'new_col_names' list
film_revenue_df.columns = new_col_names

In [13]:
#reordering the columns to organize more relevant/similar data together
film_revenue_df = film_revenue_df[['film', 'id', 'release_date', 'original_language', 'genre_ids', 'production_countries', 
       'revenue (USD)', 'budget (USD)', 'popularity', 'vote_count', 'vote_average', 
       'release_status', 'tagline', 'plot', 'runtime (min)']]
#displaying the first 5 rows of the dataframe to verify the column names were changed and rearranged
film_revenue_df.head()

,film,id,release_date,original_language,genre_ids,production_countries,revenue (USD),budget (USD),popularity,vote_count,vote_average,release_status,tagline,plot,runtime (min)
0,Avatar,19995,2009-12-15,en,"[28, 12, 14, 878]","[{'iso_3166_1': 'US', 'name': 'United States o...",2923706026,-237000000,111.323,31119,7.582,Released,Enter the world of Pandora.,"In the 22nd century, a paraplegic Marine is di...",162
1,Avengers: Endgame,299534,2019-04-24,en,"[12, 878, 28]","[{'iso_3166_1': 'US', 'name': 'United States o...",2799439100,-356000000,202.043,25178,8.253,Released,Avenge the fallen.,After the devastating events of Avengers: Infi...,181
2,Avatar: The Way of Water,76600,2022-12-14,en,"[878, 12, 28]","[{'iso_3166_1': 'US', 'name': 'United States o...",2320250281,-460000000,169.075,11554,7.622,Released,Return to Pandora.,Set more than a decade after the events of the...,192
3,Titanic,597,1997-11-18,en,"[18, 10749]","[{'iso_3166_1': 'US', 'name': 'United States o...",2264162353,-200000000,206.498,24922,7.906,Released,Nothing on Earth could come between them.,101-year-old Rose DeWitt Bukater tells the sto...,194
4,Star Wars: The Force Awakens,140607,2015-12-15,en,"[12, 28, 878]","[{'iso_3166_1': 'US', 'name': 'United States o...",2068223624,-245000000,90.923,19088,7.300,Released,Every generation has a story.,Thirty years after defeating the Galactic Empi...,136


## Step 3: Editing Monetary Values into a More Readable Format
Currently the monetary values for the 'budget (USD)' and 'revenue (USD)' columns are displayed as a long list of numbers such as: 1453683476. In order to change the monetary values to a more readable format I will need to first convert each of the columns to a float64 data type, then format the numeric values to US currency.

In [14]:
#checking the data types for all of the columns
#looking to see what data type the 'budget (USD)'and 'revenue (USD)' columns are
film_revenue_df.dtypes

film                     object
id                        int64
release_date             object
original_language        object
genre_ids                object
production_countries     object
revenue (USD)             int64
budget (USD)              int64
popularity              float64
vote_count                int64
vote_average            float64
release_status           object
tagline                  object
plot                     object
runtime (min)             int64
dtype: object

In [15]:
#using .astype() to convert the 'revenue (USD)' and 'budget (USD)' columns to float64
film_revenue_df['revenue (USD)'] = film_revenue_df['revenue (USD)'].astype(float)
film_revenue_df['budget (USD)'] = film_revenue_df['budget (USD)'].astype(float)

#define a function to convert the values to a monetary format
def format_currency(value):
    #using a format specifier to convert the numeric value to a currency string
    return '${:,.2f}'.format(value)

#the map() function applies the format_currency function to each value in the 'revenue (USD)' and 'budget (USD)' columns
film_revenue_df['revenue (USD)'] = film_revenue_df['revenue (USD)'].map(format_currency)
film_revenue_df['budget (USD)'] = film_revenue_df['budget (USD)'].map(format_currency)

In [16]:
film_revenue_df.head()

,film,id,release_date,original_language,genre_ids,production_countries,revenue (USD),budget (USD),popularity,vote_count,vote_average,release_status,tagline,plot,runtime (min)
0,Avatar,19995,2009-12-15,en,"[28, 12, 14, 878]","[{'iso_3166_1': 'US', 'name': 'United States o...","$2,923,706,026.00","$-237,000,000.00",111.323,31119,7.582,Released,Enter the world of Pandora.,"In the 22nd century, a paraplegic Marine is di...",162
1,Avengers: Endgame,299534,2019-04-24,en,"[12, 878, 28]","[{'iso_3166_1': 'US', 'name': 'United States o...","$2,799,439,100.00","$-356,000,000.00",202.043,25178,8.253,Released,Avenge the fallen.,After the devastating events of Avengers: Infi...,181
2,Avatar: The Way of Water,76600,2022-12-14,en,"[878, 12, 28]","[{'iso_3166_1': 'US', 'name': 'United States o...","$2,320,250,281.00","$-460,000,000.00",169.075,11554,7.622,Released,Return to Pandora.,Set more than a decade after the events of the...,192
3,Titanic,597,1997-11-18,en,"[18, 10749]","[{'iso_3166_1': 'US', 'name': 'United States o...","$2,264,162,353.00","$-200,000,000.00",206.498,24922,7.906,Released,Nothing on Earth could come between them.,101-year-old Rose DeWitt Bukater tells the sto...,194
4,Star Wars: The Force Awakens,140607,2015-12-15,en,"[12, 28, 878]","[{'iso_3166_1': 'US', 'name': 'United States o...","$2,068,223,624.00","$-245,000,000.00",90.923,19088,7.300,Released,Every generation has a story.,Thirty years after defeating the Galactic Empi...,136


## Step 4: Format Data into a More Readable Format
The 'original_language' column currently has abbreviations for each language and could be more clear.
* For example replacing the value 'en' with 'English'.

The 'production-countries' column contains a variety of unnecessary information/characters.
* For example, one data value contains: 'iso_3166_1': 'JP', 'name': 'Japan'}, {'iso_3166_1': 'US', 'name': 'United States of America'
* For this column, we only need the names of the countries, so we will extract just the country names. 


In [17]:
#replacing 'en' with 'English' in the 'original_language' column
film_revenue_df['original_language'].replace('en', 'English', inplace=True)

C:\Users\freck\AppData\Local\Temp\ipykernel_13216\3914411351.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  film_revenue_df['original_language'].replace('en', 'English', inplace=True)


In [18]:
film_revenue_df['production_countries'].value_counts()

production_countries
[{'iso_3166_1': 'US', 'name': 'United States of America'}]                                                    15
[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]     2
[{'iso_3166_1': 'US', 'name': 'United States of America'}, {'iso_3166_1': 'GB', 'name': 'United Kingdom'}]     1
[]                                                                                                             1
[{'iso_3166_1': 'JP', 'name': 'Japan'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]              1
Name: count, dtype: int64

In [19]:
#extracting country names from the 'production_countries' column
#.apply() applies the lambda function to each element in the 'production_countries' column
film_revenue_df['production_countries'] = film_revenue_df['production_countries'].apply(
    lambda x: ', '.join([country['name'] for country in x]))

In [20]:
film_revenue_df.head()

,film,id,release_date,original_language,genre_ids,production_countries,revenue (USD),budget (USD),popularity,vote_count,vote_average,release_status,tagline,plot,runtime (min)
0,Avatar,19995,2009-12-15,English,"[28, 12, 14, 878]","United States of America, United Kingdom","$2,923,706,026.00","$-237,000,000.00",111.323,31119,7.582,Released,Enter the world of Pandora.,"In the 22nd century, a paraplegic Marine is di...",162
1,Avengers: Endgame,299534,2019-04-24,English,"[12, 878, 28]",United States of America,"$2,799,439,100.00","$-356,000,000.00",202.043,25178,8.253,Released,Avenge the fallen.,After the devastating events of Avengers: Infi...,181
2,Avatar: The Way of Water,76600,2022-12-14,English,"[878, 12, 28]",United States of America,"$2,320,250,281.00","$-460,000,000.00",169.075,11554,7.622,Released,Return to Pandora.,Set more than a decade after the events of the...,192
3,Titanic,597,1997-11-18,English,"[18, 10749]",United States of America,"$2,264,162,353.00","$-200,000,000.00",206.498,24922,7.906,Released,Nothing on Earth could come between them.,101-year-old Rose DeWitt Bukater tells the sto...,194
4,Star Wars: The Force Awakens,140607,2015-12-15,English,"[12, 28, 878]",United States of America,"$2,068,223,624.00","$-245,000,000.00",90.923,19088,7.300,Released,Every generation has a story.,Thirty years after defeating the Galactic Empi...,136


## Step 5: Formatting the "release_date" column into DateTime
Currently the "release_date" column is type 'object'. We will want to convert this to DateTime format so that future data analysis, visualization, retrieval, and storage can be standardized and consistent. 

In [21]:
#using pd.to_datetime() to convert from type object to DateTime
film_revenue_df['release_date'] = pd.to_datetime(film_revenue_df['release_date'])

In [22]:
#displaying the data types for each of the columns to see that 'release_date' has changed to datetime
film_revenue_df.dtypes

film                            object
id                               int64
release_date            datetime64[ns]
original_language               object
genre_ids                       object
production_countries            object
revenue (USD)                   object
budget (USD)                    object
popularity                     float64
vote_count                       int64
vote_average                   float64
release_status                  object
tagline                         object
plot                            object
runtime (min)                    int64
dtype: object

## Step 6: Creating a "Year" Column
The other two datasets for this project include a 'Year' column that can potentially be used as a primary key when combining the datasets. This dataset contains the movie release date as a full date, and we want to extract just the year to create a 'Year' column. 

In [23]:
#using .dt.year to extract the year value from the 'release_date' column DateTime value
#creating a new column titled 'year' that will store the new year values
film_revenue_df['year'] = film_revenue_df['release_date'].dt.year

In [24]:
#reordering the header (columns) so that the new 'year' column is next to 'release_date' 
film_revenue_df = film_revenue_df[['film', 'id', 'year', 'release_date', 'original_language', 'genre_ids', 'production_countries', 
       'revenue (USD)', 'budget (USD)', 'popularity', 'vote_count', 'vote_average', 
       'release_status', 'tagline', 'plot', 'runtime (min)']]

## Displaying the Human Readable Dataset:

In [25]:
#displaying all 20 rows of the dataset
film_revenue_df

,film,id,year,release_date,original_language,genre_ids,production_countries,revenue (USD),budget (USD),popularity,vote_count,vote_average,release_status,tagline,plot,runtime (min)
0,Avatar,19995,2009,2009-12-15,English,"[28, 12, 14, 878]","United States of America, United Kingdom","$2,923,706,026.00","$-237,000,000.00",111.323,31119,7.582,Released,Enter the world of Pandora.,"In the 22nd century, a paraplegic Marine is di...",162
1,Avengers: Endgame,299534,2019,2019-04-24,English,"[12, 878, 28]",United States of America,"$2,799,439,100.00","$-356,000,000.00",202.043,25178,8.253,Released,Avenge the fallen.,After the devastating events of Avengers: Infi...,181
2,Avatar: The Way of Water,76600,2022,2022-12-14,English,"[878, 12, 28]",United States of America,"$2,320,250,281.00","$-460,000,000.00",169.075,11554,7.622,Released,Return to Pandora.,Set more than a decade after the events of the...,192
3,Titanic,597,1997,1997-11-18,English,"[18, 10749]",United States of America,"$2,264,162,353.00","$-200,000,000.00",206.498,24922,7.906,Released,Nothing on Earth could come between them.,101-year-old Rose DeWitt Bukater tells the sto...,194
4,Star Wars: The Force Awakens,140607,2015,2015-12-15,English,"[12, 28, 878]",United States of America,"$2,068,223,624.00","$-245,000,000.00",90.923,19088,7.300,Released,Every generation has a story.,Thirty years after defeating the Galactic Empi...,136
5,Avengers: Infinity War,299536,2018,2018-04-25,English,"[12, 28, 878]",United States of America,"$2,052,415,039.00","$-300,000,000.00",370.572,29220,8.246,Released,An entire universe. Once and for all.,As the Avengers and their allies have continue...,149
6,Best Of Joy,1326885,2024,2024-12-25,English,"[18, 10402]",,"$2,000,000,000.00","$-1,000.00",0.000,0,0.000,Released,"It was the Best of Joy, it was the Worst of Joy",When his biggest show inches closer and closer...,120
7,Spider-Man: No Way Home,634649,2021,2021-12-15,English,"[28, 12, 878]",United States of America,"$1,921,847,111.00","$-200,000,000.00",280.575,19693,8.000,Released,The Multiverse unleashed.,Peter Parker is unmasked and no longer able to...,148
8,Jurassic World,135397,2015,2015-06-06,English,"[28, 12, 878, 53]",United States of America,"$1,671,537,444.00","$-150,000,000.00",98.471,20079,6.689,Released,The park is open.,Twenty-two years after the events of Jurassic ...,124
9,The Lion King,420818,2019,2019-07-12,English,"[16, 12, 18, 10751]",United States of America,"$1,663,000,000.00","$-260,000,000.00",118.265,9796,7.113,Released,The King has Returned.,"Simba idolizes his father, King Mufasa, and ta...",118


## Summary:

&emsp; **What changes were made to the data?** I made 6 cleaning/formatting changes to the flat file data. These include finding missing values and duplicates, replacing and rearranging headers, editing monetary values into a more readable format, formatting other column data into a more readable format, formatting the "release_date" column into DateTime, and creating a "Year" column.  

&emsp; **Did you make any assumptions in cleaning/transforming the data?** One of the assumptions that I made was that the 'budget' and 'revenue' columns were values in US dollars (USD). A second assumption that I made is that the API request pulled the correct information. For example, that my specific loop and if statement to pull only the movies with the highest revenue worked properly. 

&emsp; **What risks could be created based on the transformations done?** The main risk is that there might be an error in my coding that I did not notice, which might cause the data to be inaccurately calculated or represented. Another risk is that one of the assumptions I made when transforming the data might be incorrect, causing the data to now be represented incorrectly. 

&emsp; **How was your data sourced/verified for credibility?** The data source is the TMDB API (The Movie Database API), which is a service that allows developers to retrieve information such as movie details, cast and crew information, ratings, reviews, images, and more. Data from APIs can be used to build applications, websites, or any other projects that require movie-related information. TMDB API appears to have a credible standing, supported by its usage, licensing terms, functionality, and user base. For example, the TMDB API offers free usage for non-commercial purposes, with attribution required when using the data and/or images. This indicates that the API has clear terms of use and licensing guidelines, contributing to its credibility.

&emsp; **Was your data acquired in an ethical way?** The data the TMDB API is readily available to the public. The data on TMDB is sourced from its community of users, who have contributed data on movies and TV shows from around the world. As long as TMDB is following any copyright or creative commons requirements for film related data, then there would be no reason to believe this data was not sourced in an ethical way. 

&emsp; **Are there any legal or regulatory guidelines for your data or project topic?** I have not found any legal or regulatory guidelines regarding the use of film data specifically. However, if any personal identifying information is used, it must be handled in accordance with current data privacy laws and in some cases, consent may be required to use personal data. The TMDB API is free to use for non-commercial purposes, and users are required to attribute TMDB as the source of the data and/or images when using the API. So, am compliant as long as I am referencing TMDB as my source and not trying to make money off of the utilization of the data. 

&emsp; **How would you mitigate any of the ethical implications you have identified?** In order to mitigate any ethical implications when working with this data, it is important that I am aware of any potential biases and that I do not attempt to preserve any existing biases. Transparency is essential, and it is important to take steps to prevent data manipulation or misrepresentation that could lead to misleading conclusions or unfair treatment. I will need to exercise caution and responsibility in the interpretation and reporting of the results to avoid any harm or negative consequences to the individuals and film companies this data is about.

In [26]:
film_revenue_df.dtypes

film                            object
id                               int64
year                             int32
release_date            datetime64[ns]
original_language               object
genre_ids                       object
production_countries            object
revenue (USD)                   object
budget (USD)                    object
popularity                     float64
vote_count                       int64
vote_average                   float64
release_status                  object
tagline                         object
plot                            object
runtime (min)                    int64
dtype: object

In [27]:
film_revenue_df['year'] = pd.to_numeric(film_revenue_df['year'], errors='coerce').astype('Int64')

In [28]:
film_revenue_df.rename(columns={'year': 'Year'}, inplace=True)

In [29]:
film_revenue_df.rename(columns={'film': 'Film'}, inplace=True)

In [30]:
film_revenue_df.dtypes

Film                            object
id                               int64
Year                             Int64
release_date            datetime64[ns]
original_language               object
genre_ids                       object
production_countries            object
revenue (USD)                   object
budget (USD)                    object
popularity                     float64
vote_count                       int64
vote_average                   float64
release_status                  object
tagline                         object
plot                            object
runtime (min)                    int64
dtype: object

In [32]:
#checking if there are duplicate 'id' column values to see if it can be used as a primary key
duplicate_id = film_revenue_df[film_revenue_df.duplicated(subset='Film', keep=False)]['id'].unique()
print(duplicate_id)

[]


# Uncomment below to Save to CSV again...

In [34]:
#film_revenue_df.to_csv('film_revenue2_data.csv', index=False)